# Faza 0 — Pozyskanie danych FSA/OWI i budowa par

## Cel projektu "Decydujący moment"

Ten notebook realizuje **pierwszy krok Fazy 0**: pobranie metadanych z kolekcji FSA/OWI Library of Congress, zbudowanie par "wybrane vs odrzucone z tej samej sceny", i **wizualną weryfikację na małej próbce** zanim zbudujemy pełny pipeline.

### Logika eksperymentu

Kolekcja FSA/OWI (1935-1944, ~175 000 negatywów) ma wbudowaną strukturę redaktorskiego wyboru:
- **Wydrukowane** klatki — wybrane przez Roya Strykera (pełny tytuł w metadanych)
- **Niewydrukowane** klatki — odrzucone, często z captionem `[Untitled photo, possibly related to: ...]`

Sąsiednie klatki z tej samej rolki (sąsiednie call numbers) pokazują tę samą scenę w różnych momentach. Para (wybrana, odrzucona) z tej samej sceny izoluje **czysty sygnał momentu** — wszystkie konfaundy (B&W, grain, epoka, fotograf, scena) są stałe.

### Co robi ten notebook

1. Odpytuje API loc.gov o próbkę kolekcji FSA/OWI (JSON, bez klucza)
2. Analizuje strukturę odpowiedzi (diagnostyka — co faktycznie zwraca API)
3. Identyfikuje klatki wydrukowane vs untitled (proxy dla wybrane vs odrzucone)
4. Grupuje sąsiednie klatki w pary przez call number
5. Pobiera obrazy przez IIIF i wyświetla pary do wizualnej oceny

### Uczciwość metodologiczna

Operacjonalizujemy "decydujący moment" jako *"klatka z pełnym tytułem redaktorskim vs sąsiedni untitled wariant tej samej sceny"*. To proxy — Stryker wybierał też za przydatność dokumentacyjną i jakość techniczną, nie tylko za moment. Caption "Untitled" jako sygnał odrzucenia to kolejne przybliżenie. Mimo to jest to znacznie czystszy sygnał niż jakiekolwiek alternatywne źródło.


## 0. Setup


In [1]:
import time
import json
import re
from pathlib import Path
from collections import defaultdict

import requests
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

# Konfiguracja
OUTPUT_DIR = Path('fsa_data')
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'images').mkdir(exist_ok=True)

# Rate limiting — KLUCZOWE, by nie dostać blokady od loc.gov
REQUEST_DELAY = 2.0   # sekundy między żądaniami API

# User-Agent — grzeczność wobec serwera (identyfikacja projektu badawczego)
HEADERS = {
    'User-Agent': 'DecisiveMoment-Research/1.0 (academic project; contact: your-email@example.com)'
}

print('Setup gotowy')
print(f'Output: {OUTPUT_DIR.resolve()}')
print(f'Rate limit: {REQUEST_DELAY}s między żądaniami')


Setup gotowy
Output: /content/fsa_data
Rate limit: 2.0s między żądaniami


In [13]:
import requests, json, time

HEADERS = {'User-Agent': 'DecisiveMoment-Research/1.0 (academic)'}
url = 'https://www.loc.gov/collections/fsa-owi-black-and-white-negatives/'
resp = requests.get(url, params={'fo':'json','c':3,'at':'results,pagination'}, headers=HEADERS, timeout=30)
data = resp.json()

# Pokaż WSZYSTKIE klucze pierwszego rekordu z pełnymi wartościami
rec = data['results'][0]
print("=== WSZYSTKIE KLUCZE REKORDU ===")
for k in sorted(rec.keys()):
    v = rec[k]
    if isinstance(v, (list, dict)):
        print(f"\n{k} ({type(v).__name__}, len={len(v)}):")
        print("  " + json.dumps(v, indent=2, ensure_ascii=False)[:500])
    else:
        print(f"\n{k}: {repr(v)[:300]}")

=== WSZYSTKIE KLUCZE REKORDU ===

access_restricted: False

aka (list, len=8):
  [
  "http://www.loc.gov/pictures/collection/fsa/item/2013651650/",
  "https://hdl.loc.gov/loc.pnp/cph.3c21074",
  "http://www.loc.gov/item/2013651650/",
  "http://www.loc.gov/pictures/item/2013651650/",
  "https://hdl.loc.gov/loc.pnp/ppmsc.00190",
  "http://www.loc.gov/resource/ppmsc.00190/",
  "http://www.loc.gov/resource/cph.3c21074/",
  "http://lccn.loc.gov/2013651650"
]

campaigns (list, len=0):
  []

date: '1943-01-01'

dates (list, len=1):
  [
  "1943"
]

description (list, len=1):
  [
  "1 photographic print : gelatin silver."
]

digitized: True

extract_timestamp: '2021-09-01T20:10:02.135Z'

group (list, len=6):
  [
  "fsa",
  "catalog-split-13",
  "catalog",
  "fsa-owi-black-and-white-negatives",
  "main-catalog-split-13",
  "main-catalog"
]

hassegments: False

id: 'http://www.loc.gov/item/2013651650/'

image_url (list, len=4):
  [
  "https://tile.loc.gov/storage-services/service/pnp/ppmsc/00100/

## 1. Pomocnicza funkcja do odpytywania API loc.gov

API loc.gov jest publiczne (bez klucza), ale ma rate limiting. Funkcja poniżej dodaje opóźnienie, retry przy błędach, i obsługę paginacji.


In [2]:
def loc_api_get(url, params=None, max_retries=3):
    """Odpytuje API loc.gov z rate limiting i retry.

    Zwraca sparsowany JSON lub None przy błędzie.
    """
    if params is None:
        params = {}
    params['fo'] = 'json'  # zawsze żądaj JSON

    for attempt in range(max_retries):
        try:
            time.sleep(REQUEST_DELAY)  # rate limiting PRZED żądaniem
            resp = requests.get(url, params=params, headers=HEADERS, timeout=30)

            if resp.status_code == 200:
                if 'json' in resp.headers.get('content-type', ''):
                    return resp.json()
                else:
                    print(f'  ⚠ Odpowiedź nie-JSON (content-type: {resp.headers.get("content-type")})')
                    return None
            elif resp.status_code == 429:
                # Too Many Requests — czekaj dłużej
                wait = REQUEST_DELAY * (attempt + 2) * 3
                print(f'  ⚠ Rate limit (429), czekam {wait:.0f}s...')
                time.sleep(wait)
            else:
                print(f'  ⚠ HTTP {resp.status_code} (próba {attempt+1}/{max_retries})')
                time.sleep(REQUEST_DELAY * 2)
        except requests.exceptions.RequestException as e:
            print(f'  ⚠ Błąd sieci: {e} (próba {attempt+1}/{max_retries})')
            time.sleep(REQUEST_DELAY * 2)

    print(f'  ✗ Nie udało się pobrać po {max_retries} próbach: {url}')
    return None

# Test na małej próbce — pobierz 5 itemów z kolekcji FSA/OWI
FSA_COLLECTION_URL = 'https://www.loc.gov/collections/fsa-owi-black-and-white-negatives/'

print('Testuję połączenie z API loc.gov...')
test = loc_api_get(FSA_COLLECTION_URL, params={'c': 5, 'at': 'results,pagination'})

if test:
    print('✅ API odpowiada')
    print(f'  Klucze najwyższego poziomu: {list(test.keys())}')
    if 'pagination' in test:
        print(f'  Pagination: {test["pagination"]}')
else:
    print('✗ API nie odpowiada — sprawdź połączenie lub zwiększ REQUEST_DELAY')


Testuję połączenie z API loc.gov...
✅ API odpowiada
  Klucze najwyższego poziomu: ['pagination', 'results']
  Pagination: {'current': 1, 'first': None, 'from': 1, 'last': 'https://www.loc.gov/collections/fsa-owi-black-and-white-negatives/?c=5&fo=json&sp=4', 'next': 'https://www.loc.gov/collections/fsa-owi-black-and-white-negatives/?c=5&fo=json&sp=2', 'of': 171054, 'page_list': [{'number': 1, 'url': None}, {'number': 2, 'url': 'https://www.loc.gov/collections/fsa-owi-black-and-white-negatives/?c=5&fo=json&sp=2'}, {'number': 3, 'url': 'https://www.loc.gov/collections/fsa-owi-black-and-white-negatives/?c=5&fo=json&sp=3'}, {'number': '...', 'url': 'https://www.loc.gov/collections/fsa-owi-black-and-white-negatives/?c=5&fo=json&sp=4'}], 'perpage': 5, 'perpage_options': [25, 50, 100, 150], 'previous': None, 'results': '1 - 5', 'to': 5, 'total': 34211}


## 2. Diagnostyka — co faktycznie zwraca API?

Zanim zbudujemy pipeline, obejrzyjmy **rzeczywistą strukturę** jednego rekordu. Struktura JSON API może się różnić od dokumentacji, więc patrzymy na konkretne pola.


In [3]:
if test and 'results' in test and len(test['results']) > 0:
    sample = test['results'][0]
    print('Klucze pojedynczego rekordu "results":')
    for key in sorted(sample.keys()):
        val = sample[key]
        # Skrócony podgląd wartości
        if isinstance(val, (list, dict)):
            preview = f'{type(val).__name__} (len={len(val)})'
        else:
            preview = str(val)[:80]
        print(f'  {key:25} : {preview}')

    print('\n' + '='*60)
    print('Pełny pierwszy rekord (JSON):')
    print('='*60)
    print(json.dumps(sample, indent=2, ensure_ascii=False)[:2000])
else:
    print('Brak wyników do diagnostyki — uruchom komórkę 1 najpierw')


Klucze pojedynczego rekordu "results":
  access_restricted         : False
  aka                       : list (len=8)
  campaigns                 : list (len=0)
  date                      : 1943-01-01
  dates                     : list (len=1)
  description               : list (len=1)
  digitized                 : True
  extract_timestamp         : 2021-09-01T20:10:02.135Z
  group                     : list (len=6)
  hassegments               : False
  id                        : http://www.loc.gov/item/2013651650/
  image_url                 : list (len=4)
  index                     : 1
  item                      : dict (len=34)
  language                  : list (len=1)
  mime_type                 : list (len=3)
  number                    : list (len=5)
  number_former_id          : list (len=3)
  number_lccn               : list (len=1)
  number_source_modified    : list (len=1)
  online_format             : list (len=1)
  original_format           : list (len=1)
  partof      

## 3. Pobierz większą próbkę metadanych

Pobieramy kilka stron wyników (kilkaset rekordów) do analizy. Paginujemy ostrożnie z rate limiting.

**Uwaga**: dla pełnego datasetu zwiększ `MAX_PAGES`, ale na start wystarczy kilka stron do weryfikacji podejścia.


In [8]:
MAX_PAGES = 20          # liczba stron do pobrania (na start mało)
RESULTS_PER_PAGE = 100  # max 100

# Pola które chcemy wyciągnąć (zmniejsza rozmiar odpowiedzi)
ATTRS = 'results,pagination'

all_records = []
next_url = FSA_COLLECTION_URL
next_params = {'c': RESULTS_PER_PAGE, 'at': ATTRS, 'sp': 1}

print(f'Pobieram do {MAX_PAGES} stron po {RESULTS_PER_PAGE} rekordów...')
print('(z rate limiting — to potrwa)\n')

for page in range(1, MAX_PAGES + 1):
    print(f'Strona {page}/{MAX_PAGES}...', end=' ')
    next_params['sp'] = page
    data = loc_api_get(FSA_COLLECTION_URL, params=dict(next_params))

    if not data or 'results' not in data:
        print('brak danych, przerywam')
        break

    results = data['results']
    all_records.extend(results)
    print(f'+{len(results)} rekordów (łącznie {len(all_records)})')

    # Sprawdź czy jest następna strona
    pagination = data.get('pagination', {})
    if pagination.get('next') is None:
        print('  (ostatnia strona)')
        break

print(f'\n✅ Pobrano {len(all_records)} rekordów metadanych')


Pobieram do 20 stron po 100 rekordów...
(z rate limiting — to potrwa)

Strona 1/20... +100 rekordów (łącznie 100)
Strona 2/20... +100 rekordów (łącznie 200)
Strona 3/20... +100 rekordów (łącznie 300)
Strona 4/20... +100 rekordów (łącznie 400)
Strona 5/20... +100 rekordów (łącznie 500)
Strona 6/20...   ⚠ Błąd sieci: HTTPSConnectionPool(host='www.loc.gov', port=443): Read timed out. (próba 1/3)
  ⚠ Błąd sieci: ('Connection broken: IncompleteRead(3584 bytes read, 992446 more expected)', IncompleteRead(3584 bytes read, 992446 more expected)) (próba 2/3)
+100 rekordów (łącznie 600)
Strona 7/20... +100 rekordów (łącznie 700)
Strona 8/20...   ⚠ Błąd sieci: HTTPSConnectionPool(host='www.loc.gov', port=443): Read timed out. (próba 1/3)
  ⚠ Błąd sieci: ('Connection broken: IncompleteRead(3583 bytes read, 1032159 more expected)', IncompleteRead(3583 bytes read, 1032159 more expected)) (próba 2/3)
+100 rekordów (łącznie 800)
Strona 9/20... +100 rekordów (łącznie 900)
Strona 10/20... +100 rekordów 

## 4. Wyciągnij kluczowe pola i zidentyfikuj wydrukowane vs untitled

Z każdego rekordu wyciągamy: tytuł, call number, link do obrazu (IIIF), oraz **flagę czy to untitled** (proxy dla odrzucone).


In [9]:
def is_untitled(title):
    """Czy tytuł sugeruje niewydrukowaną (odrzuconą) klatkę?

    FSA untitled mają zazwyczaj: '[Untitled photo, possibly related to: ...]'
    """
    if not title:
        return True
    t = title.lower()
    return 'untitled' in t or 'possibly related' in t

def extract_related_scene(title):
    """Z 'Untitled photo, possibly related to: X' wyciąga X (opis sceny).

    To pozwala grupować untitled z ich 'rodzicem' wydrukowanym.
    """
    if not title:
        return None
    m = re.search(r'possibly related to:?\s*(.+?)[\]\.]?$', title, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return None

def extract_fields(record):
    """Wyciąga ustandaryzowane pola z rekordu API.

    Defensywnie — API może mieć różne struktury, więc sprawdzamy alternatywy.
    """
    # Tytuł
    title = record.get('title', '')

    # Call number — może być w różnych miejscach
    call_number = None
    for field in ['call_number', 'number_lccn', 'shelf_id']:
        if record.get(field):
            val = record[field]
            call_number = val[0] if isinstance(val, list) else val
            break

    # Link do obrazu — preferuj image_url, fallback do innych
    image_url = None
    if record.get('image_url'):
        img = record['image_url']
        image_url = img[0] if isinstance(img, list) else img

    # ID itemu (do dalszego odpytania jeśli trzeba)
    item_id = record.get('id', '')

    # Data
    date = record.get('date', '')

    return {
        'title': title,
        'call_number': call_number,
        'image_url': image_url,
        'item_id': item_id,
        'date': date,
        'is_untitled': is_untitled(title),
        'related_scene': extract_related_scene(title),
    }

# Przetwórz wszystkie rekordy
records_df = pd.DataFrame([extract_fields(r) for r in all_records])

print(f'Przetworzono {len(records_df)} rekordów\n')
print('Rozkład wydrukowane vs untitled:')
print(records_df['is_untitled'].value_counts())
print(f'\n  Wydrukowane (wybrane): {(~records_df["is_untitled"]).sum()}')
print(f'  Untitled (odrzucone):  {records_df["is_untitled"].sum()}')

# Ile ma call number i image_url
print(f'\n  Z call_number: {records_df["call_number"].notna().sum()}')
print(f'  Z image_url:   {records_df["image_url"].notna().sum()}')

records_df.head(10)


Przetworzono 2000 rekordów

Rozkład wydrukowane vs untitled:
is_untitled
False    1998
True        2
Name: count, dtype: int64

  Wydrukowane (wybrane): 1998
  Untitled (odrzucone):  2

  Z call_number: 2000
  Z image_url:   1999


,title,call_number,image_url,item_id,date,is_untitled,related_scene
0,"[Gordon Parks, Farm Security Administration/Of...",2013651650,https://tile.loc.gov/storage-services/service/...,http://www.loc.gov/item/2013651650/,1943-01-01,False,None
1,"Mary Anderson, head of Women's Bureau of the U...",90706083,https://tile.loc.gov/storage-services/service/...,http://www.loc.gov/item/90706083/,1942-01-01,False,None
2,Arkansas family of migrant workers now in Cali...,2024666076,https://tile.loc.gov/storage-services/service/...,http://www.loc.gov/item/2024666076/,1936,False,None
3,Members of the Woman's Land Army,2024665999,https://tile.loc.gov/storage-services/service/...,http://www.loc.gov/item/2024665999/,1943,False,None
4,A tank production plant,2024666000,https://tile.loc.gov/storage-services/service/...,http://www.loc.gov/item/2024666000/,1942,False,None
5,Production line inside a tank production plant,2024665792,https://tile.loc.gov/storage-services/service/...,http://www.loc.gov/item/2024665792/,1942,False,None
6,"Pittsburgh vicinity, Pa., Nov. 1942. Village n...",2024665913,https://tile.loc.gov/storage-services/service/...,http://www.loc.gov/item/2024665913/,1942,False,None
7,"Pennsylvania Dutch barn, Lancaster vicinity, Pa.",2024666008,https://tile.loc.gov/storage-services/service/...,http://www.loc.gov/item/2024666008/,1938,False,None
8,"Pennsylvania Dutch barn, Lancaster vicinity, Pa.",2024666009,https://tile.loc.gov/storage-services/service/...,http://www.loc.gov/item/2024666009/,1938,False,None
9,"Harvest time in Middletown Valley, Frederick (...",2024665758,https://tile.loc.gov/storage-services/service/...,http://www.loc.gov/item/2024665758/,1930,False,None


## 5. Buduj pary przez powiązanie scen

Dwie strategie grupowania w pary:

**Strategia A — przez `related_scene`**: untitled klatki mówią wprost "possibly related to: X". Jeśli znajdziemy wydrukowaną klatkę o tytule podobnym do X, mamy parę.

**Strategia B — przez sąsiedztwo call number**: sąsiednie call numbers = sąsiednie klatki tej samej rolki. (Wymaga sortowania po call number.)

Zaczynamy od Strategii A (prostsza, bezpośredni sygnał z metadanych).


In [10]:
# Strategia A: dopasuj untitled do wydrukowanych przez opis sceny

printed = records_df[~records_df['is_untitled']].copy()
untitled = records_df[records_df['is_untitled']].copy()

print(f'Wydrukowane: {len(printed)}, Untitled: {len(untitled)}\n')

def normalize_text(s):
    """Normalizacja do porównywania tytułów."""
    if not s:
        return ''
    s = s.lower().strip()
    s = re.sub(r'[\[\]\.\,]', '', s)
    s = re.sub(r'\s+', ' ', s)
    return s

# Zbuduj indeks wydrukowanych po znormalizowanym tytule
printed['title_norm'] = printed['title'].apply(normalize_text)

pairs = []
for _, u_row in untitled.iterrows():
    scene = u_row['related_scene']
    if not scene:
        continue
    scene_norm = normalize_text(scene)
    if not scene_norm:
        continue

    # Szukaj wydrukowanej której tytuł zawiera/jest zawarty w opisie sceny
    for _, p_row in printed.iterrows():
        p_norm = p_row['title_norm']
        if not p_norm:
            continue
        # Dopasowanie: opis sceny untitled pasuje do tytułu wydrukowanej
        if scene_norm in p_norm or p_norm in scene_norm:
            if u_row['image_url'] and p_row['image_url']:
                pairs.append({
                    'chosen_title': p_row['title'],
                    'chosen_url': p_row['image_url'],
                    'chosen_call': p_row['call_number'],
                    'rejected_title': u_row['title'],
                    'rejected_url': u_row['image_url'],
                    'rejected_call': u_row['call_number'],
                    'scene': scene,
                })
            break  # pierwsza pasująca wydrukowana wystarczy

pairs_df = pd.DataFrame(pairs)
print(f'✅ Znaleziono {len(pairs_df)} par (wybrane vs odrzucone z tej samej sceny)\n')

if len(pairs_df) > 0:
    print('Przykładowe pary:')
    for i, row in pairs_df.head(3).iterrows():
        print(f'\n[{i}] Scena: {row["scene"][:60]}')
        print(f'    Wybrana:   {row["chosen_title"][:70]}')
        print(f'    Odrzucona: {row["rejected_title"][:70]}')
else:
    print('⚠ Brak par w tej próbce. Możliwe przyczyny:')
    print('  - Za mała próbka (untitled i ich rodzice na różnych stronach)')
    print('  - Inna struktura tytułów niż zakładana')
    print('  → Zwiększ MAX_PAGES lub przejdź do Strategii B (call number)')


Wydrukowane: 1998, Untitled: 2

✅ Znaleziono 0 par (wybrane vs odrzucone z tej samej sceny)

⚠ Brak par w tej próbce. Możliwe przyczyny:
  - Za mała próbka (untitled i ich rodzice na różnych stronach)
  - Inna struktura tytułów niż zakładana
  → Zwiększ MAX_PAGES lub przejdź do Strategii B (call number)


## 6. Pobierz obrazy przez IIIF i wyświetl pary

Dla znalezionych par pobieramy obrazy w rozsądnym rozmiarze (przez IIIF) i wyświetlamy obok siebie do **wizualnej oceny jakości par**.


In [11]:
def to_iiif_url(image_url, size='!400,400'):
    """Konwertuje URL obrazu LoC na IIIF z określonym rozmiarem.

    size='!400,400' = wpasuj w 400x400 zachowując proporcje.
    Jeśli URL już jest pełnym linkiem do jpg, zwraca go bez zmian.
    """
    if not image_url:
        return None
    # Jeśli to już bezpośredni link do obrazu, użyj go
    if image_url.endswith('.jpg') or image_url.endswith('.jpeg'):
        return image_url
    return image_url

def fetch_image(url, size='!400,400'):
    """Pobiera obraz, zwraca PIL.Image lub None."""
    iiif_url = to_iiif_url(url, size)
    if not iiif_url:
        return None
    try:
        time.sleep(REQUEST_DELAY)
        resp = requests.get(iiif_url, headers=HEADERS, timeout=30)
        if resp.status_code == 200:
            return Image.open(BytesIO(resp.content))
        else:
            print(f'    ⚠ HTTP {resp.status_code} dla {iiif_url[:60]}')
    except Exception as e:
        print(f'    ⚠ Błąd pobierania: {e}')
    return None

# Wyświetl pierwsze N par
N_PAIRS_TO_SHOW = min(4, len(pairs_df))

if N_PAIRS_TO_SHOW > 0:
    fig, axes = plt.subplots(N_PAIRS_TO_SHOW, 2, figsize=(10, 5 * N_PAIRS_TO_SHOW))
    if N_PAIRS_TO_SHOW == 1:
        axes = axes.reshape(1, 2)

    for i in range(N_PAIRS_TO_SHOW):
        row = pairs_df.iloc[i]
        print(f'Pobieram parę {i+1}/{N_PAIRS_TO_SHOW}...')

        img_chosen = fetch_image(row['chosen_url'])
        img_rejected = fetch_image(row['rejected_url'])

        if img_chosen:
            axes[i, 0].imshow(img_chosen)
        axes[i, 0].set_title(f'WYBRANA\n{row["chosen_title"][:50]}', fontsize=9, color='green')
        axes[i, 0].axis('off')

        if img_rejected:
            axes[i, 1].imshow(img_rejected)
        axes[i, 1].set_title(f'ODRZUCONA\n{row["rejected_title"][:50]}', fontsize=9, color='red')
        axes[i, 1].axis('off')

    plt.suptitle('Pary FSA/OWI — wybrane (Stryker) vs odrzucone z tej samej sceny',
                 fontsize=13, fontweight='bold', y=1.005)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'sample_pairs.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('\n✅ Zapisano podgląd: fsa_data/sample_pairs.png')
else:
    print('Brak par do wyświetlenia — patrz diagnostyka w komórce 5')


Brak par do wyświetlenia — patrz diagnostyka w komórce 5


## 7. Zapisz wyniki i podsumowanie


In [ ]:
# Zapisz metadane i pary do CSV
records_df.to_csv(OUTPUT_DIR / 'fsa_records.csv', index=False)
if len(pairs_df) > 0:
    pairs_df.to_csv(OUTPUT_DIR / 'fsa_pairs.csv', index=False)

print('='*60)
print('FAZA 0 — PODSUMOWANIE PRÓBKI')
print('='*60)
print(f'Pobrane rekordy:        {len(records_df)}')
print(f'  Wydrukowane:          {(~records_df["is_untitled"]).sum()}')
print(f'  Untitled:             {records_df["is_untitled"].sum()}')
print(f'Znalezione pary:        {len(pairs_df)}')
print(f'\nPliki zapisane w {OUTPUT_DIR.resolve()}:')
print(f'  fsa_records.csv  — wszystkie rekordy')
if len(pairs_df) > 0:
    print(f'  fsa_pairs.csv    — pary wybrane/odrzucone')
    print(f'  sample_pairs.png — podgląd wizualny')

print('\n' + '='*60)
print('CO DALEJ')
print('='*60)
print("""
1. OCEŃ WIZUALNIE pary w sample_pairs.png:
   - Czy wybrana i odrzucona to FAKTYCZNIE ta sama scena?
   - Czy widać różnicę w "momencie" (gest, ekspresja, kompozycja)?
   - Jeśli pary są dobre → podejście działa, skaluj MAX_PAGES
   - Jeśli pary mieszają różne sceny → potrzebny filtr CLIP similarity

2. SKALUJ jeśli próbka wygląda dobrze:
   - Zwiększ MAX_PAGES (np. 50-100 stron = 5000-10000 rekordów)
   - Uwaga na czas: rate limiting 2s × liczba żądań

3. DODAJ FILTR CLIP (następny notebook):
   - Cosine similarity między wybraną a odrzuconą
   - Zostaw pary o wysokiej similarity (ta sama scena, inny moment)
   - To eliminuje fałszywe pary (różne sceny z tej samej rolki)

4. ALTERNATYWA — Photogrammar CSV:
   - Jeśli API jest za wolne, pobierz gotowy CSV z Photogrammar
   - github.com/nolauren/Rtutorial_photogrammar
   - ~82k rekordów z linkami, bez potrzeby paginacji API
""")


## 8. (Opcjonalnie) Strategia B — pary przez sąsiedztwo call number

Jeśli Strategia A (przez `related_scene`) dała mało par, ta komórka próbuje grupować przez **sąsiednie call numbers**. Sąsiednie numery = sąsiednie klatki tej samej rolki.

Call numbers FSA mają format typu `LC-USF34-001234-D` — gdzie liczba rośnie wzdłuż rolki. Sortując po niej i biorąc sąsiadów, dostajemy klatki tej samej sesji.


In [12]:
def parse_call_number(call):
    """Wyciąga numeryczną część call number FSA do sortowania.

    Np. 'LC-USF34-001234-D' -> (prefix='LC-USF34', num=1234)
    Pozwala znaleźć sąsiednie klatki tej samej rolki.
    """
    if not call:
        return None, None
    # Szukaj wzorca: prefix - liczba - suffix
    m = re.search(r'(.*?)(\d{4,})', str(call))
    if m:
        prefix = m.group(1)
        num = int(m.group(2))
        return prefix, num
    return None, None

# Parsuj call numbers
records_df['call_prefix'] = records_df['call_number'].apply(lambda c: parse_call_number(c)[0])
records_df['call_num'] = records_df['call_number'].apply(lambda c: parse_call_number(c)[1])

valid_calls = records_df[records_df['call_num'].notna()].copy()
print(f'Rekordy z parsowalnym call number: {len(valid_calls)}/{len(records_df)}')

if len(valid_calls) > 0:
    valid_calls = valid_calls.sort_values(['call_prefix', 'call_num']).reset_index(drop=True)

    # Szukaj par: wydrukowana + sąsiednia untitled (call_num różni się o 1-3)
    neighbor_pairs = []
    for i in range(len(valid_calls)):
        row_i = valid_calls.iloc[i]
        if row_i['is_untitled']:
            continue  # szukamy wydrukowanej jako kotwicy

        # Sprawdź sąsiadów w obrębie ±3 klatek
        for j in range(max(0, i-3), min(len(valid_calls), i+4)):
            if j == i:
                continue
            row_j = valid_calls.iloc[j]
            if not row_j['is_untitled']:
                continue  # szukamy untitled jako pary
            # Ta sama rolka (prefix) i bliski numer?
            if (row_i['call_prefix'] == row_j['call_prefix'] and
                abs(row_i['call_num'] - row_j['call_num']) <= 3):
                if row_i['image_url'] and row_j['image_url']:
                    neighbor_pairs.append({
                        'chosen_title': row_i['title'],
                        'chosen_url': row_i['image_url'],
                        'chosen_call': row_i['call_number'],
                        'rejected_title': row_j['title'],
                        'rejected_url': row_j['image_url'],
                        'rejected_call': row_j['call_number'],
                        'call_distance': abs(row_i['call_num'] - row_j['call_num']),
                    })

    neighbor_df = pd.DataFrame(neighbor_pairs)
    # Usuń duplikaty
    if len(neighbor_df) > 0:
        neighbor_df = neighbor_df.drop_duplicates(subset=['chosen_call', 'rejected_call'])

    print(f'\n✅ Strategia B znalazła {len(neighbor_df)} par przez sąsiedztwo call number')
    if len(neighbor_df) > 0:
        neighbor_df.to_csv(OUTPUT_DIR / 'fsa_pairs_by_callnum.csv', index=False)
        print('   Zapisano: fsa_data/fsa_pairs_by_callnum.csv')
        print('\n   Porównaj liczbę par: Strategia A vs B i wybierz lepszą,')
        print('   lub połącz obie i odfiltruj duplikaty.')
else:
    print('Brak parsowalnych call numbers — sprawdź format w diagnostyce (komórka 2)')


Rekordy z parsowalnym call number: 2000/2000

✅ Strategia B znalazła 12 par przez sąsiedztwo call number
   Zapisano: fsa_data/fsa_pairs_by_callnum.csv

   Porównaj liczbę par: Strategia A vs B i wybierz lepszą,
   lub połącz obie i odfiltruj duplikaty.
